In [1]:
from PIL import Image
from transformers import AutoProcessor, LlavaForConditionalGeneration, AutoProcessor, LlavaImageProcessor, LlamaTokenizer
import pandas as pd
import numpy as np
import torch
import io

In [2]:
data  = pd.read_parquet('/projectnb/cs598/projects/cool_proj/dataset', engine = 'pyarrow')

In [3]:
import os

print(os.getcwd())

/projectnb/cs598/projects/cool_proj/SuryaWorking/Pause_Finetune


In [4]:
model_id = 'llava-hf/llava-1.5-7b-hf'
cache_dir = '../SuryaWorking/LLaVa'
ft_model_path = 'epoch_2'
torch.cuda.empty_cache()

model = LlavaForConditionalGeneration.from_pretrained(ft_model_path).to(0).eval()
processor = AutoProcessor.from_pretrained(model_id, cache_dir = cache_dir)
tokenizer = LlamaTokenizer.from_pretrained(ft_model_path,  use_fast = False)
processor.tokenizer = tokenizer


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.48, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [5]:
def convert_to_bytesio(image_bytes):
    return np.asarray([Image.open(io.BytesIO(im_bytes)) for im_bytes in image_bytes], dtype=np.uint8)
    
data['image_bytes'] = data['image_bytes'].apply(convert_to_bytesio)

In [7]:
torch.cuda.empty_cache()
from tqdm import tqdm
import re
import random
def extract_ans(pred_text):
    pred_text = pred_text.split("ASSISTANT:", 1)[1].strip()
    matches = re.findall(r"\[(.*?)\]", pred_text)
    
    return matches

def predict_all(dataset, device='cuda'):
    predictions = []
    n_correct = 0
    for i in (pbar := tqdm(range(len(dataset['question'])))):
        input_image = dataset['image_bytes'][i]
        question = dataset['question'][i]
        answer_choices = dataset['answers'][i]
        shuffled_choices = answer_choices.copy()
        random.shuffle(shuffled_choices)
        input_prompt = f'''
    Instructions: You have to answer the following question using the given options.
    Question: {question}
    Options: {shuffled_choices}
    Enclose the answer in brackets like [answer].
    <pause>'''

        content = [{"type": "text", "text": f"{input_prompt}"}]
        for j in range(len(input_image)):
            content.append({"type": "image", f"image {j+1}" : f"{input_image[j]}"})
            # print(content)
        conversation = [
                {
            
                  "role": "user",
                  "content": content,
                },
            ]
        prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)
        inputs = processor(images=input_image, text=prompt, return_tensors='pt').to(0, torch.float16)
        output = model.generate(**inputs, max_new_tokens=100, do_sample=False)
        prediction_text = processor.decode(output[0][2:], skip_special_tokens=True)
        
        # print(prediction_text)
        prediction = extract_ans(prediction_text)
        # print("Q: ",question)
        # print("options: ",answer_choices)
        # print("answer: ",prediction_text)
        # print("Correct_a: ",dataset['correct_answer'][i])
        # print(conversation)
        predictions.append({
            "question": question,
            "answers": answer_choices,
            "pred_text": prediction_text,
            "prediction": prediction,
            "correct_answer": dataset['correct_answer'][i]
        })
        print("choices: ", shuffled_choices)
        print("model pred: ",prediction)
        print("correct_ans: ",dataset['correct_answer'][i])
        print('pepepe')
        
        if len(prediction) > 0:
            if prediction[0] in dataset['correct_answer'][i] or dataset['correct_answer'][i] in prediction[0]:
                print('yay?')
                n_correct += 1
  
                
        
            
        pbar.set_description(f'{n_correct} correct')
    
    return predictions

predictions = predict_all(data)

0 correct:   0%|          | 1/4001 [00:05<6:20:32,  5.71s/it]

choices:  ['Chair was moved left and towards the camera in the first frame'
 'Chair was moved right and away from the camera in the first frame']
model pred:  ['Chair was moved right and away from the camera in the first frame', 'Chair was moved right and away from the camera in the first frame', 'Chair was moved right and away from the camera in the first frame', 'Chair was moved right and away from the camera in the first frame', 'Chair was moved right and away from the camera in the first frame']
correct_ans:  Chair was moved left and towards the camera in the first frame
pepepe


0 correct:   0%|          | 2/4001 [00:11<6:06:45,  5.50s/it]

choices:  ['rotated left' 'did not move']
model pred:  ['rotated left', 'rotated left', 'rotated left', 'rotated left', 'rotated left', 'rotated left', 'rotated left', 'rotated left', 'rotated left', 'rotated left', 'rotated left', 'rotated left', 'rotated left', 'rotated left', 'rotated left', 'rotated left', 'rotated left', 'rotated left', 'rotated left']
correct_ans:  did not move
pepepe


0 correct:   0%|          | 3/4001 [00:15<5:41:18,  5.12s/it]

...

choices:  ['Closer' 'Further']
model pred:  ['Closer', 'Further', 'Further', 'Further', 'Further', 'Closer', 'Further', 'Further', 'Further', 'Closer', 'Further', 'Further', 'Further', 'Closer', 'Further', 'Further', 'Closer', 'Further', 'Further']
correct_ans:  Closer
pepepe
yay?


1700 correct: 100%|█████████▉| 4000/4001 [5:26:55<00:04,  4.68s/it]

choices:  ['Further' 'Closer']
model pred:  ['Closer', 'Closer', 'Closer', 'Closer', 'Closer', 'Closer', 'Closer', 'Closer', 'Closer', 'Closer', 'Closer', 'Closer', 'Closer', 'Closer', 'Closer', 'Closer', 'Closer', 'Closer', 'Closer']
correct_ans:  Closer
pepepe
yay?


1701 correct: 100%|██████████| 4001/4001 [5:26:59<00:00,  4.90s/it]

choices:  ['Closer' 'Further']
model pred:  ['Closer', 'Further', 'Closer', 'Further', 'Closer', 'Further', 'Closer', 'Closer', 'Further', 'Closer', 'Further', 'Closer', 'Closer', 'Further', 'Closer', 'Further', 'Closer', 'Closer', 'Closer']
correct_ans:  Closer
pepepe
yay?


In [11]:
import json
import numpy as np

for entry in predictions:
    for key, value in entry.items():
        if isinstance(value, np.ndarray):
            entry[key] = value.tolist()
        elif isinstance(value, list):
            # If it's a list, make sure every element is not ndarray
            entry[key] = [v.tolist() if isinstance(v, np.ndarray) else v for v in value]
with open("predictions_pauseft.json", "w", encoding="utf-8") as f:
    json.dump(predictions, f, indent=4, ensure_ascii=False)
